# TrackNetV3 fine-tune (free GPU)

Run the cells top to bottom. You only upload one file: **`finetune_data.zip`** (in your *Line Calls* folder).
At the end you download the fine-tuned model and drop it back into your TrackNetV3 folder.

**First, turn on the GPU:** menu **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save.**


### 1. Check the GPU is on


In [ ]:
!nvidia-smi -L
import torch; print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

If it says `CUDA available: False` or shows no GPU, set the runtime to **T4 GPU** (see top), then **Runtime -> Restart session** and run this again.


### 2. Get the TrackNetV3 code


In [ ]:
!git clone -q https://github.com/qaz812345/TrackNetV3.git
%cd TrackNetV3
!pip install -q parse

### 3. Download the pretrained model (~130 MB, automatic)


In [ ]:
!pip install -q -U gdown
!gdown -q 1CfzE87a0f6LhBp0kniSl1-89zaLCZ8cA -O ckpts.zip
!mkdir -p ckpts && unzip -o -q ckpts.zip -d ckpts
import glob; print('checkpoints found:', glob.glob('ckpts/**/*.pt', recursive=True))

If the line above is empty (Google Drive occasionally blocks the auto-download), upload your local `ckpts/TrackNet_best.pt` and `ckpts/InpaintNet_best.pt` manually with the file panel on the left instead.


### 4. Upload your clips
Run the cell, click **Choose Files**, and pick **`finetune_data.zip`** from your *Line Calls* folder.


In [ ]:
from google.colab import files
up = files.upload()                       # choose finetune_data.zip
print('uploaded:', list(up))

### 5. Prepare the data (memory-safe)
Extracts frames, estimates the background from a sample of frames, and forms the validation split. Finishes in under a minute and prints `frames 1_00 750`, `frames 1_01 187`, then `done`.

*(This replaces the repo's `preprocess.py`, whose median step loads every frame at full resolution and runs out of memory on a long single-rally clip like this one.)*


In [ ]:
# Memory-safe data prep (replaces preprocess.py, whose median step OOMs on long rallies)
import os, glob, shutil, cv2, numpy as np

!rm -rf data
!unzip -o -q finetune_data.zip -d .

DATA, MATCH = "data", "match24"
TM = f"{DATA}/train/{MATCH}"

def extract(video, outdir):
    os.makedirs(outdir, exist_ok=True)
    cap = cv2.VideoCapture(video); i = 0
    while True:
        ok, fr = cap.read()
        if not ok: break
        cv2.imwrite(f"{outdir}/{i}.png", fr); i += 1
    cap.release(); return i

def rally_median(video, n=80):                    # sample n frames -> low RAM
    cap = cv2.VideoCapture(video)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    keep = set(np.linspace(0, max(total-1, 0), min(n, total)).astype(int).tolist())
    fr, i = [], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if i in keep: fr.append(frame)
        i += 1
    cap.release()
    return np.median(np.array(fr), 0)[..., ::-1]  # BGR -> RGB

for v in sorted(glob.glob(f"{TM}/video/*.mp4")):
    name = os.path.splitext(os.path.basename(v))[0]
    print("frames", name, extract(v, f"{TM}/frame/{name}"))

meds = [rally_median(v) for v in sorted(glob.glob(f"{TM}/video/*.mp4"))]
np.savez(f"{TM}/median.npz", median=np.median(np.array(meds), 0))

rallies = sorted(glob.glob(f"{TM}/video/*.mp4"))
val = os.path.splitext(os.path.basename(rallies[-1]))[0]
VM = f"{DATA}/val/{MATCH}"
for s in ["video", "csv", "frame"]: os.makedirs(f"{VM}/{s}", exist_ok=True)
shutil.move(f"{TM}/video/{val}.mp4",    f"{VM}/video/{val}.mp4")
shutil.move(f"{TM}/csv/{val}_ball.csv", f"{VM}/csv/{val}_ball.csv")
shutil.move(f"{TM}/frame/{val}",        f"{VM}/frame/{val}")
np.savez(f"{VM}/median.npz", median=rally_median(f"{VM}/video/{val}.mp4"))
print("done -- train:", [os.path.basename(x) for x in glob.glob(f"{TM}/video/*.mp4")], "| val:", val)

### 6. Fine-tune on the GPU
A full run is a few minutes on a T4. You should see `Device: cuda`, `train samples: 743`, then a `train ... val ...` line per epoch. Watch the **train loss** go down. If you hit an out-of-memory error, change `--batch_size 8` to `4`.


In [ ]:
import glob
pt = sorted(glob.glob('ckpts/**/TrackNet_best.pt', recursive=True))[0]
print('starting from:', pt)
!python finetune.py --pretrained "{pt}" --device cuda --epochs 12 --batch_size 8 --lr 1e-4

### 7. Download your fine-tuned model


In [ ]:
from google.colab import files
files.download('exp_finetune/TrackNet_finetune_best.pt')

### Done
Put the downloaded **`TrackNet_finetune_best.pt`** into your local `TrackNetV3/exp_finetune/` folder. You can then use it anywhere the original model is used by pointing `--tracknet_file` at it. To label and add more clips later, rebuild `finetune_data.zip` the same way and re-run this notebook.
